In [ ]:
!pip install ultralytics open_clip_torch transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [open_clip_torch]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import random
import shutil

BASE_PATH = "/content/drive/MyDrive/copro/data"

RAW_PATH = os.path.join(BASE_PATH, "raw")
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")

classes = ["neem", "tulasi"]

for cls in classes:

    raw_class = os.path.join(RAW_PATH, cls)
    train_class = os.path.join(TRAIN_PATH, cls)
    val_class = os.path.join(VAL_PATH, cls)

    os.makedirs(train_class, exist_ok=True)
    os.makedirs(val_class, exist_ok=True)

    images = [
        f for f in os.listdir(raw_class)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)

    split_index = int(0.8 * len(images))

    train_images = images[:split_index]
    val_images = images[split_index:]

    # Copy training images
    for img in train_images:
        shutil.copy(
            os.path.join(raw_class, img),
            os.path.join(train_class, img)
        )

    # Copy validation images
    for img in val_images:
        shutil.copy(
            os.path.join(raw_class, img),
            os.path.join(val_class, img)
        )

    print(f"{cls} → Train: {len(train_images)} | Val: {len(val_images)}")

print("✅ 80:20 dataset split completed successfully")

neem → Train: 116 | Val: 30
tulasi → Train: 116 | Val: 30
✅ 80:20 dataset split completed successfully


In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolov8m-cls.yaml")

model.train(
    data="/content/drive/MyDrive/copro/data",
    epochs=25,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu"
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLOv8m-cls summary: 80 layers, 17,053,336 parameters, 17,053,336 gradients, 41.9 GFLOPs
Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/copro/data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epoc

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f82b41e5c50>
curves: []
curves_results: []
fitness: 0.9942084848880768
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9884169697761536, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9942084848880768}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 2.3828004208479725, 'inference': 7.9627662548255564, 'loss': 0.0005003050211816464, 'postprocess': 0.0009241930526714057}
top1: 0.9884169697761536
top5: 1.0

In [ ]:
import os
import json
import cv2
import numpy as np
from ultralytics import YOLO
import open_clip
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

In [ ]:
class QualityMetrics:

    def compute(self, image_path):

        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        blur = cv2.Laplacian(img, cv2.CV_64F).var()
        brightness = np.mean(img) / 255
        contrast = np.std(img) / 255

        return {
            "blur_score": round(float(blur),3),
            "brightness": round(float(brightness),3),
            "contrast": round(float(contrast),3)
        }

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

class YOLOExtractor:

    def __init__(self):

        # Your trained classification model
        self.model = YOLO("/content/runs/classify/train/weights/best.pt")

        self.conf = 0.05


    def extract(self, image_path):

        objects = []

        image = cv2.imread(image_path)

        if image is None:
            return objects

        h, w = image.shape[:2]
        image_area = h * w

        results = self.model.predict(
            source=image_path,
            conf=self.conf,
            imgsz=640,
            verbose=False
        )

        for r in results:

            if r.probs is None:
                continue

            class_id = int(r.probs.top1)
            confidence = float(r.probs.top1conf)

            label = self.model.names[class_id]

            # Whole image as bounding box
            x1, y1, x2, y2 = 0, 0, w, h

            bbox_area = w * h
            area_ratio = bbox_area / image_area

            # Calculate green ratio
            hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

            lower_green = np.array([35,40,40])
            upper_green = np.array([85,255,255])

            mask = cv2.inRange(hsv, lower_green, upper_green)

            green_pixels = np.sum(mask > 0)
            total_pixels = h * w

            green_ratio = green_pixels / total_pixels

            # Leaf stage estimation
            if green_ratio > 0.6:
                leaf_stage = "Mature"
            elif green_ratio > 0.3:
                leaf_stage = "Young"
            else:
                leaf_stage = "Old"

            objects.append({
                "label": label,
                "confidence": round(confidence,3),
                "bbox": [x1,y1,x2,y2],
                "area_ratio": round(float(area_ratio),3),
                "green_ratio": round(float(green_ratio),3),
                "leaf_stage": leaf_stage
            })

        return objects

In [ ]:
class CLIPExtractor:

    def __init__(self):

        self.model,_,self.preprocess = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained="openai"
        )

        self.model.eval()

    def extract(self,image_path):

        image = self.preprocess(Image.open(image_path)).unsqueeze(0)

        with torch.no_grad():
            embedding = self.model.encode_image(image)

        return embedding.cpu().numpy().flatten().tolist()

In [ ]:
class BLIPExtractor:

    def __init__(self):

        self.processor = BlipProcessor.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )

        self.model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )

    def extract(self,image_path):

        image = Image.open(image_path).convert("RGB")

        inputs = self.processor(image, return_tensors="pt")

        out = self.model.generate(**inputs)

        caption = self.processor.decode(out[0], skip_special_tokens=True)

        return caption

In [ ]:
class FeatureFusion:

    def fuse(self, image_name, quality, clip_emb, caption, objects,
             color_mode, original_size, resized_size):

        return {
            "image": image_name,

            "image_properties": {
                "color_mode": color_mode,
                "original_size": original_size,
                "resized_size": resized_size
            },

            "quality_metrics": quality,

            "semantic_features": {
                "clip_embedding": clip_emb
            },

            "descriptive_features": {
                "caption": caption
            },

            "object_features": {
                "detected_objects": objects,
                "object_count": len(objects)
            }
        }

In [ ]:
quality_checker = QualityMetrics()

yolo_extractor = YOLOExtractor()

clip_extractor = CLIPExtractor()

blip_extractor = BLIPExtractor()

fusion_engine = FeatureFusion()

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  990MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [ ]:
import os
import json

DATASET_DIR = "/content/drive/MyDrive/copro/data/raw"
OUTPUT_DIR = "/content/drive/MyDrive/copro/output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import cv2
import os

final_dataset = []

for root, dirs, files in os.walk(DATASET_DIR):

    for img_name in files:

        if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(root, img_name)

        print("Processing:", img_name)

        # Load image
        image = cv2.imread(image_path)
        if image is None:
            continue

        # Image properties
        original_size = (image.shape[1], image.shape[0])   # (width, height)

        resized_image = cv2.resize(image, (640, 640))
        resized_size = (640, 640)

        color_mode = "RGB"

        # Existing pipeline
        quality = quality_checker.compute(image_path)
        clip_emb = clip_extractor.extract(image_path)
        caption = blip_extractor.extract(image_path)
        objects = yolo_extractor.extract(image_path)

        # Updated fusion call
        structured = fusion_engine.fuse(
            img_name,
            quality,
            clip_emb,
            caption,
            objects,
            color_mode,
            original_size,
            resized_size
        )

        final_dataset.append(structured)

Processing: 1803.jpg


model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1963: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Processing: 1812.jpg
Processing: 1787.jpg
Processing: 1830.jpg
Processing: 1831.jpg
Processing: 1834.jpg
Processing: 1806.jpg
Processing: 1828.jpg
Processing: 1810.jpg
Processing: 1788.jpg
Processing: 1807.jpg
Processing: 1805.jpg
Processing: 1818.jpg
Processing: 1794.jpg
Processing: 1808.jpg
Processing: 1789.jpg
Processing: 1786.jpg
Processing: 1819.jpg
Processing: 1802.jpg
Processing: 1832.jpg
Processing: 1804.jpg
Processing: 1822.jpg
Processing: 1799.jpg
Processing: 1785.jpg
Processing: 1790.jpg
Processing: 1797.jpg
Processing: 1781.jpg
Processing: 1816.jpg
Processing: 1833.jpg
Processing: 1811.jpg
Processing: 1823.jpg
Processing: 1782.jpg
Processing: 1835.jpg
Processing: 1825.jpg
Processing: 1793.jpg
Processing: 1824.jpg
Processing: 1815.jpg
Processing: 1817.jpg
Processing: 1795.jpg
Processing: 1796.jpg
Processing: 1827.jpg
Processing: 1821.jpg
Processing: 1792.jpg
Processing: 1780.jpg
Processing: 1783.jpg
Processing: 1801.jpg
Processing: 1826.jpg
Processing: 1836.jpg
Processing: 1

In [ ]:
OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "plant_multimodal_dataset1.json"
)

with open(OUTPUT_JSON, "w") as f:
    json.dump(final_dataset, f, indent=4)

print("JSON saved at:", OUTPUT_JSON)
print("Total images processed:", len(final_dataset))

JSON saved at: /content/drive/MyDrive/copro/output/plant_multimodal_dataset1.json
Total images processed: 292


In [ ]:
import os
import json
import cv2
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO
import open_clip
from transformers import BlipProcessor, BlipForConditionalGeneration

# --- 1. Device and Memory Config ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
torch.cuda.empty_cache()

# --- 2. Adaptive Enhancement Engine ---
class AdaptiveEnhancer:
    @staticmethod
    def enhance(image, quality_metrics):
        # OpenCV read as BGR
        enhanced = image.copy()

        # 1. Low Contrast correction: Apply CLAHE in LAB space
        if quality_metrics["contrast"] < 0.18:
            lab = cv2.cvtColor(enhanced, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            cl = clahe.apply(l)
            lab = cv2.merge((cl, a, b))
            enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

        # 2. Blur correction: Apply Unsharp Masking
        if quality_metrics["blur_score"] < 150.0:
            gaussian = cv2.GaussianBlur(enhanced, (0, 0), 2.0)
            enhanced = cv2.addWeighted(enhanced, 1.5, gaussian, -0.5, 0)

        # 3. Low Brightness correction: Non-linear Gamma Correction
        if quality_metrics["brightness"] < 0.35:
            inv_gamma = 1.0 / 1.4
            table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
            enhanced = cv2.LUT(enhanced, table)

        return enhanced

# --- 3. Tri-Model Predictors ---
class TriModelPredictor:
    def __init__(self):
        # YOLO
        self.yolo_model = YOLO("/content/runs/classify/train/weights/best.pt")

        # CLIP
        self.clip_model, _, self.clip_preprocess = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained="openai"
        )
        self.clip_model = self.clip_model.to(device)
        self.clip_model.eval()
        self.tokenizer = open_clip.get_tokenizer("ViT-B-32")

        # BLIP
        self.blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        self.blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

        self.classes = ["neem", "tulasi"]
        self.clip_prompts = [f"a photo of a {cls} leaf" for cls in self.classes]
        self.text_tokens = self.tokenizer(self.clip_prompts).to(device)

    def predict_yolo(self, image_bgr):
        # Run prediction on standard size 640
        results = self.yolo_model.predict(source=image_bgr, imgsz=640, verbose=False)
        if not results or results[0].probs is None:
            return None, 0.0
        probs = results[0].probs
        top1_idx = int(probs.top1)
        conf = float(probs.top1conf)
        return self.yolo_model.names[top1_idx], conf

    def predict_clip(self, image_bgr):
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(image_rgb)
        processed_img = self.clip_preprocess(pil_img).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = self.clip_model.encode_image(processed_img)
            image_features /= image_features.norm(dim=-1, keepdim=True)

            text_features = self.clip_model.encode_text(self.text_tokens)
            text_features /= text_features.norm(dim=-1, keepdim=True)

            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            values, indices = similarity[0].topk(1)

        pred_idx = indices[0].item()
        conf = values[0].item()
        return self.classes[pred_idx], conf, image_features.cpu().numpy().flatten().tolist()

    def predict_blip(self, image_bgr):
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(image_rgb)

        inputs = self.blip_processor(pil_img, return_tensors="pt").to(device)
        with torch.no_grad():
            out = self.blip_model.generate(**inputs)
        caption = self.blip_processor.decode(out[0], skip_special_tokens=True).lower()

        detected_class = None
        for cls in self.classes:
            if cls in caption:
                detected_class = cls
                break
        return detected_class, caption

# --- 4. Agentic Consensus Engine ---
class AgenticOrchestrator:
    def __init__(self, predictor, quality_checker):
        self.predictor = predictor
        self.quality_checker = quality_checker

    def process_image(self, image_path, max_retries=2):
        original_bgr = cv2.imread(image_path)
        if original_bgr is None:
            return None

        current_bgr = original_bgr.copy()
        retry_count = 0

        while retry_count <= max_retries:
            # Quality Metrics extraction
            q_metrics = self.quality_checker.compute(image_path if retry_count == 0 else "_enhanced_temp_filename_")
            # Overwrite compute inputs dynamically with current BGR in-memory properties
            img_gray = cv2.cvtColor(current_bgr, cv2.COLOR_BGR2GRAY)
            q_metrics = {
                "blur_score": round(float(cv2.Laplacian(img_gray, cv2.CV_64F).var()), 3),
                "brightness": round(float(np.mean(img_gray) / 255.0), 3),
                "contrast": round(float(np.std(img_gray) / 255.0), 3)
            }

            # Execute Tri-Model Inference
            yolo_cls, yolo_conf = self.predictor.predict_yolo(current_bgr)
            clip_cls, clip_conf, clip_emb = self.predictor.predict_clip(current_bgr)
            blip_cls, blip_caption = self.predictor.predict_blip(current_bgr)

            # Determine consensus agreement
            predictions = {"yolo": yolo_cls, "clip": clip_cls, "blip": blip_cls}
            valid_preds = [v for v in predictions.values() if v is not None]

            consensus_label = None
            agreement_ratio = "0/3"

            if len(valid_preds) > 0:
                from collections import Counter
                count = Counter(valid_preds)
                most_common, occurrence = count.most_common(1)[0]
                consensus_label = most_common
                agreement_ratio = f"{occurrence}/3"

            # Evaluation Policy Rules
            state = "REPROCESS"
            if agreement_ratio == "3/3" and yolo_conf >= 0.75 and clip_conf >= 0.75:
                state = "ACCEPT"
            elif agreement_ratio == "2/3" and yolo_cls == consensus_label and yolo_conf >= 0.80:
                state = "ACCEPT"
            elif retry_count == max_retries:
                state = "FLAG_FOR_REVIEW"

            if state == "ACCEPT":
                return {
                    "status": "ACCEPT",
                    "consensus_label": consensus_label,
                    "agreement_ratio": agreement_ratio,
                    "retry_count": retry_count,
                    "quality_metrics": q_metrics,
                    "clip_embedding": clip_emb,
                    "yolo_confidence": round(yolo_conf, 3),
                    "clip_confidence": round(clip_conf, 3),
                    "caption": blip_caption
                }
            elif state == "FLAG_FOR_REVIEW":
                return {
                    "status": "FLAG_FOR_REVIEW",
                    "consensus_label": consensus_label,
                    "agreement_ratio": agreement_ratio,
                    "retry_count": retry_count,
                    "quality_metrics": q_metrics,
                    "clip_embedding": clip_emb,
                    "yolo_confidence": round(yolo_conf, 3) if yolo_conf else 0.0,
                    "clip_confidence": round(clip_conf, 3) if clip_conf else 0.0,
                    "caption": blip_caption
                }

            # State is REPROCESS: Apply Adaptive enhancement
            current_bgr = AdaptiveEnhancer.enhance(current_bgr, q_metrics)
            retry_count += 1
            torch.cuda.empty_cache()

        return None

Using device: cuda


In [ ]:
# --- 5. Main Execution Pipeline & Performance Metrics ---

DATASET_DIR = "/content/drive/MyDrive/copro/data/raw"
OUTPUT_DIR = "/content/drive/MyDrive/copro/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fix the process_image method of AgenticOrchestrator dynamically
def fixed_process_image(self, image_path, max_retries=2):
    original_bgr = cv2.imread(image_path)
    if original_bgr is None:
        return None

    current_bgr = original_bgr.copy()
    retry_count = 0

    while retry_count <= max_retries:
        # Directly extract gray image from in-memory BGR to get quality metrics
        img_gray = cv2.cvtColor(current_bgr, cv2.COLOR_BGR2GRAY)
        q_metrics = {
            "blur_score": round(float(cv2.Laplacian(img_gray, cv2.CV_64F).var()), 3),
            "brightness": round(float(np.mean(img_gray) / 255.0), 3),
            "contrast": round(float(np.std(img_gray) / 255.0), 3)
        }

        # Execute Tri-Model Inference
        yolo_cls, yolo_conf = self.predictor.predict_yolo(current_bgr)
        clip_cls, clip_conf, clip_emb = self.predictor.predict_clip(current_bgr)
        blip_cls, blip_caption = self.predictor.predict_blip(current_bgr)

        # Determine consensus agreement
        predictions = {"yolo": yolo_cls, "clip": clip_cls, "blip": blip_cls}
        valid_preds = [v for v in predictions.values() if v is not None]

        consensus_label = None
        agreement_ratio = "0/3"

        if len(valid_preds) > 0:
            from collections import Counter
            count = Counter(valid_preds)
            most_common, occurrence = count.most_common(1)[0]
            consensus_label = most_common
            agreement_ratio = f"{occurrence}/3"

        # Evaluation Policy Rules
        state = "REPROCESS"
        if agreement_ratio == "3/3" and yolo_conf >= 0.75 and clip_conf >= 0.75:
            state = "ACCEPT"
        elif agreement_ratio == "2/3" and yolo_cls == consensus_label and yolo_conf >= 0.80:
            state = "ACCEPT"
        elif retry_count == max_retries:
            state = "FLAG_FOR_REVIEW"

        if state == "ACCEPT":
            return {
                "status": "ACCEPT",
                "consensus_label": consensus_label,
                "agreement_ratio": agreement_ratio,
                "retry_count": retry_count,
                "quality_metrics": q_metrics,
                "clip_embedding": clip_emb,
                "yolo_confidence": round(yolo_conf, 3),
                "clip_confidence": round(clip_conf, 3),
                "caption": blip_caption
            }
        elif state == "FLAG_FOR_REVIEW":
            return {
                "status": "FLAG_FOR_REVIEW",
                "consensus_label": consensus_label,
                "agreement_ratio": agreement_ratio,
                "retry_count": retry_count,
                "quality_metrics": q_metrics,
                "clip_embedding": clip_emb,
                "yolo_confidence": round(yolo_conf, 3) if yolo_conf else 0.0,
                "clip_confidence": round(clip_conf, 3) if clip_conf else 0.0,
                "caption": blip_caption
            }

        # State is REPROCESS: Apply Adaptive enhancement
        current_bgr = AdaptiveEnhancer.enhance(current_bgr, q_metrics)
        retry_count += 1
        torch.cuda.empty_cache()

    return None

# Patch the method onto AgenticOrchestrator
AgenticOrchestrator.process_image = fixed_process_image

# Initialize dependencies
predictor_engine = TriModelPredictor()
orchestrator = AgenticOrchestrator(predictor_engine, quality_checker)

accepted_dataset = []
flagged_dataset = []

metrics = {
    "total": 0,
    "first_pass_accept": 0,
    "recovered": 0,
    "flagged": 0
}

# Walk and execute
for root, dirs, files in os.walk(DATASET_DIR):
    for img_name in files:
        if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(root, img_name)
        print(f"Evaluating: {img_name}...")

        res = orchestrator.process_image(image_path, max_retries=2)
        if res is None:
            continue

        metrics["total"] += 1
        record = {
            "image": img_name,
            "consensus_label": res["consensus_label"],
            "agreement_ratio": res["agreement_ratio"],
            "retry_count": res["retry_count"],
            "quality_metrics": res["quality_metrics"],
            "yolo_confidence": res["yolo_confidence"],
            "clip_confidence": res["clip_confidence"],
            "caption": res["caption"],
            "clip_embedding": res["clip_embedding"]
        }

        if res["status"] == "ACCEPT":
            accepted_dataset.append(record)
            if res["retry_count"] == 0:
                metrics["first_pass_accept"] += 1
            else:
                metrics["recovered"] += 1
        else:
            flagged_dataset.append(record)
            metrics["flagged"] += 1

# --- Save Results ---
with open(os.path.join(OUTPUT_DIR, "agentic_structured_dataset.json"), "w") as f:
    json.dump(accepted_dataset, f, indent=4)

with open(os.path.join(OUTPUT_DIR, "flagged_for_review.json"), "w") as f:
    json.dump(flagged_dataset, f, indent=4)

# Calculate recovery performance metrics
recovery_rate = (metrics["recovered"] / (metrics["recovered" ] + metrics["flagged"])) * 100 if (metrics["recovered"] + metrics["flagged"]) > 0 else 0.0

print("\n" + "="*40)
print("   RESEARCH PIPELINE EXPERIMENT SUMMARY")
print("="*40)
print(f"Total Images Evaluated:         {metrics['total']}")
print(f"Accepted (First Pass):          {metrics['first_pass_accept']}")
print(f"Recovered via Adaptive Loop:    {metrics['recovered']}")
print(f"Flagged for Manual Review:      {metrics['flagged']}")
print(f"Self-Correction Recovery Rate:  {recovery_rate:.2f}%")
print("="*40)

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Evaluating: 1803.jpg...


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1963: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Evaluating: 1812.jpg...
Evaluating: 1787.jpg...
Evaluating: 1830.jpg...
Evaluating: 1831.jpg...
Evaluating: 1834.jpg...
Evaluating: 1806.jpg...
Evaluating: 1828.jpg...
Evaluating: 1810.jpg...
Evaluating: 1788.jpg...
Evaluating: 1807.jpg...
Evaluating: 1805.jpg...
Evaluating: 1818.jpg...
Evaluating: 1794.jpg...
Evaluating: 1808.jpg...
Evaluating: 1789.jpg...
Evaluating: 1786.jpg...
Evaluating: 1819.jpg...
Evaluating: 1802.jpg...
Evaluating: 1832.jpg...
Evaluating: 1804.jpg...
Evaluating: 1822.jpg...
Evaluating: 1799.jpg...
Evaluating: 1785.jpg...
Evaluating: 1790.jpg...
Evaluating: 1797.jpg...
Evaluating: 1781.jpg...
Evaluating: 1816.jpg...
Evaluating: 1833.jpg...
Evaluating: 1811.jpg...
Evaluating: 1823.jpg...
Evaluating: 1782.jpg...
Evaluating: 1835.jpg...
Evaluating: 1825.jpg...
Evaluating: 1793.jpg...
Evaluating: 1824.jpg...
Evaluating: 1815.jpg...
Evaluating: 1817.jpg...
Evaluating: 1795.jpg...
Evaluating: 1796.jpg...
Evaluating: 1827.jpg...
Evaluating: 1821.jpg...
Evaluating: 1792